# Portfolio Analysis

This script is V2 version of my Portfolio Analysis script that takes my stock data and creates dataframes to provide detailed information on my portfolio. This update was made after the Robinhood API was updated and is no longer usable for me. The output of this script are the following CSV data files, that are then used for my Streamlit app.
- Daily stock data --> this table will contain the daily prices of all the stocks I own and their quantities. It starts from the day I began investing on RobinHood (September 9th, 2016) until today (whenever that is). This table will be used to show my holdings over time with a focus on the profit/loss of my portfolio and specific stocks over time. 
- Current stock data --> this table will provide a single snapshot of my stock holdings today. It will be used to show the state of my portfolio as it is today. 
- Company information --> to complement these two tables above, I will also be using the RobinHood API to compile market information on these stocks that can be used to supplement my analysis.

## Import Packages and Data

In [1]:
from datetime import datetime, timedelta
import ftplib
import io
import json
import numpy as np
import pandas as pd
import requests
from tenacity import retry, stop_after_attempt, wait_fixed
import time
from typing import Set, List, Optional
import yfinance as yf

In [2]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.colheader_justify', 'center')
pd.set_option('display.precision', 2)
pd.set_option('display.float_format', '{:.4f}'.format)

In [3]:
# Read in Stock Dictionary
with open('data/stock_dictionary.json', 'r') as stock_dictionary:
    stock_dictionary = json.load(stock_dictionary)

In [4]:
# Read in previous data files
daily_stocks = pd.read_csv('data/daily_stocks.csv')
stock_summary = pd.read_csv('data/stock_summary.csv')
stock_info = pd.read_csv('data/stock_info.csv')

## Create Needed Functions 

In [5]:
def build_summary_dataframe(stock_dictionary, print_info=False):
    """
    Builds a summary DataFrame for stocks using a stock dictionary and Yahoo Finance.
    
    Returns:
        pd.DataFrame: A DataFrame with columns ordered as:
        Stock, Company, Price, Quantity, Avg_Cost, Market_Value, Percent_Change, Equity_Change, 52_Week_High, 52_Week_Low
    """
    stock_data = []

    for ticker, details in stock_dictionary.items():
        try:
            # Fetch stock data from Yahoo Finance
            stock = yf.Ticker(ticker)
            stock_info = stock.info
            splits = stock.splits

            if not splits.empty:
                splits.index = splits.index.tz_localize(None)

            # Extract required fields
            current_price = stock_info.get("currentPrice")
            company_name = stock_info.get("longName", details["stock_name"])
            percent_change = stock_info.get("52WeekChange", 0) * 100
            high_52_week = stock_info.get("fiftyTwoWeekHigh")
            low_52_week = stock_info.get("fiftyTwoWeekLow")

            # Initialize cumulative tracking variables
            total_quantity = 0.0
            total_cost = 0.0
            avg_cost = 0.0

            # Combine transactions and splits into a single timeline
            events = []
            for transaction in details["purchase_history"]:
                transaction_date = pd.to_datetime(transaction["date"]).tz_localize(None)
                events.append({
                    "type": "transaction",
                    "date": transaction_date,
                    "transaction": transaction
                })
            for split_date, ratio in splits.items():
                split_date = pd.Timestamp(split_date).tz_localize(None)
                events.append({
                    "type": "split",
                    "date": split_date,
                    "ratio": ratio
                })

            # Sort events by date
            events = sorted(events, key=lambda x: x["date"])

            # Print ticker, if prompted
            if print_info:
                print(f"Processing {ticker}:")

            # Loop over events and extract transaction data
            for event in events:
                if event["type"] == "transaction":
                    # Process transactions
                    transaction = event["transaction"]
                    transaction_date = event["date"]

                    if transaction["buy_sell"] == "buy":
                        new_quantity = float(transaction["quantity"])
                        new_cost = new_quantity * transaction["share_price"]
                        prev_avg_cost = avg_cost
                        total_quantity += new_quantity
                        total_cost += new_cost
                        avg_cost = total_cost / total_quantity if total_quantity else 0.0
                        if print_info:
                            print(
                                f"Purchase on {transaction_date.date()} of {new_quantity} shares at {transaction['share_price']:.2f}. "
                                f"\nUpdated Quantity: {total_quantity}, Updated Avg Cost: {avg_cost:.2f} (Previous Avg Cost: {prev_avg_cost:.2f})\n"
                            )
                            
                    elif transaction["buy_sell"] == "sell":
                        sell_quantity = float(transaction["quantity"])
                        if sell_quantity > total_quantity:
                            raise ValueError(f"Selling more shares than owned for {ticker} on {transaction_date}")
                        prev_quantity = total_quantity
                        total_quantity -= sell_quantity
                        total_cost -= sell_quantity * avg_cost  # Total cost decreases but avg_cost remains unchanged
                        if total_quantity <= 0:
                            total_quantity = 0.0
                            total_cost = 0.0
                            avg_cost = 0.0
                        if print_info:
                            print(
                                f"Sell on {transaction_date.date()} of {sell_quantity} shares at {transaction['share_price']:.2f}. "
                                f"\nUpdated Quantity: {total_quantity} (Previous Quantity: {prev_quantity}), Avg Cost remains {avg_cost:.2f}\n"
                            )
                            
                elif event["type"] == "split":
                    split_date = event["date"]
                    ratio = event["ratio"]
                    if total_quantity > 0:  # Apply splits only if shares are owned
                        old_quantity = total_quantity
                        old_avg_cost = avg_cost
                        total_quantity *= ratio
                        avg_cost /= ratio if ratio else 1.0
                        if print_info:
                            print(
                                f"Stock Split on {split_date.date()}: Ratio {ratio}. "
                                f"\nPrevious Quantity: {old_quantity}, Previous Avg Cost: {old_avg_cost:.2f} "
                                f"\nUpdated Quantity: {total_quantity}, Updated Avg Cost: {avg_cost:.2f}\n"
                            )

            market_value = total_quantity * current_price if current_price else 0
            equity_change = market_value - total_cost if market_value else None

            # Append stock summary
            stock_data.append([
                ticker,
                company_name,
                current_price,
                total_quantity,
                avg_cost,
                market_value,
                percent_change,
                equity_change,
                high_52_week,
                low_52_week
            ])

        except Exception as e:
            print(f"Error fetching data for {ticker}: {e}")

    # Create DataFrame
    stocks_df = pd.DataFrame(
        stock_data,
        columns=[
            "Stock",
            "Company",
            "Price",
            "Quantity",
            "Avg_Cost",
            "Market_Value",
            "Percent_Change",
            "Equity_Change",
            "52_Week_High",
            "52_Week_Low",
        ],
    )
    
    # Add in a column for portfolio diversity to stocks
    stocks_df['Portfolio_Diversity'] = round(stocks_df['Market_Value'] * 100/ sum(stocks_df['Market_Value']),2)
    # Add in column for the overall direction of the stock movement
    stocks_df['Direction'] = np.where(stocks_df['Percent_Change'] > 0, 'Up', 'Down')
    
    # Filter to just the stocks that I currently own
    filtered_stocks = stocks_df[stocks_df["Quantity"] > 0].copy()
    
    # Format the DF
    filtered_stocks['Price'] = round(filtered_stocks['Price'],2)
    filtered_stocks['Quantity'] = round(filtered_stocks['Quantity'],2)
    filtered_stocks['Avg_Cost'] = round(filtered_stocks['Avg_Cost'],2)
    filtered_stocks['Market_Value'] = round(filtered_stocks['Market_Value'],2)
    filtered_stocks['Percent_Change'] = round(filtered_stocks['Percent_Change'],2)
    filtered_stocks['Equity_Change'] = round(filtered_stocks['Equity_Change'],2)
    filtered_stocks['52_Week_High'] = round(filtered_stocks['52_Week_High'],2)
    filtered_stocks['52_Week_Low'] = round(filtered_stocks['52_Week_Low'],2)

    # Add 'Last_Updated' column
    filtered_stocks["Last_Updated"] = pd.to_datetime(datetime.today().date())

    # Sort the dataframe by the Market Value (desc)
    sorted_stocks = filtered_stocks.sort_values(by="Market_Value", ascending=False)

    return sorted_stocks

In [6]:
def format_seconds(seconds):
    minutes, sec = divmod(int(seconds), 60)
    hours, min_ = divmod(minutes, 60)
    return f"{hours:02d}:{min_:02d}:{sec:02d}"

In [7]:
def create_daily_stock_table(stock_dictionary, start_date="2016-01-01"):
    """
    Creates a detailed table of daily stock values for all stock positions.

    Returns:
        pd.DataFrame: A DataFrame with columns ordered as:
        Date, Close, Stock, Shares_Held, Avg_Cost, Equity, Market_Value, Total_Profit, Daily_Profit, Daily_Pct_Profit
    """
    daily_data = []

    for ticker, details in stock_dictionary.items():
        try:
            stock = yf.Ticker(ticker)
            historical_data = stock.history(start=start_date)
            historical_data.reset_index(inplace=True)
            historical_data["Date"] = historical_data["Date"].dt.tz_localize(None)

            total_quantity = 0
            total_cost = 0
            for transaction in details["purchase_history"]:
                transaction_date = pd.to_datetime(transaction["date"]).tz_localize(None)
                if transaction["buy_sell"] == "buy":
                    total_quantity += transaction["quantity"]
                    total_cost += transaction["quantity"] * transaction["share_price"]
                elif transaction["buy_sell"] == "sell":
                    total_quantity -= transaction["quantity"]
                    total_cost -= transaction["quantity"] * transaction["share_price"]

            avg_cost = total_cost / total_quantity if total_quantity > 0 else 0
            historical_data["Stock"] = ticker
            historical_data["Shares_Held"] = total_quantity
            historical_data["Avg_Cost"] = avg_cost
            historical_data["Equity"] = historical_data["Shares_Held"] * historical_data["Avg_Cost"]
            historical_data["Market_Value"] = historical_data["Shares_Held"] * historical_data["Close"]
            historical_data["Total_Profit"] = historical_data["Market_Value"] - historical_data["Equity"]
            historical_data["Daily_Profit"] = historical_data["Total_Profit"].diff().fillna(0)
            historical_data["Daily_Pct_Profit"] = historical_data["Close"].pct_change().fillna(0) * 100

            daily_data.append(
                historical_data[
                    [
                        "Date",
                        "Close",
                        "Stock",
                        "Shares_Held",
                        "Avg_Cost",
                        "Equity",
                        "Market_Value",
                        "Total_Profit",
                        "Daily_Profit",
                        "Daily_Pct_Profit",
                    ]
                ]
            )

        except Exception as e:
            print(f"Error fetching data for {ticker}: {e}")
            
    # Format list as a df
    daily_stocks_df = pd.concat(daily_data, ignore_index=True)

    # Format the df values
    daily_stocks_df['Close'] = round(daily_stocks_df['Close'],2)
    daily_stocks_df['Avg_Cost'] = round(daily_stocks_df['Avg_Cost'],2)
    daily_stocks_df['Market_Value'] = round(daily_stocks_df['Market_Value'],2)
    daily_stocks_df['Equity'] = round(daily_stocks_df['Equity'],2)
    daily_stocks_df['Total_Profit'] = round(daily_stocks_df['Total_Profit'],2)
    daily_stocks_df['Daily_Profit'] = round(daily_stocks_df['Daily_Profit'],2)
    daily_stocks_df['Daily_Pct_Profit'] = round(daily_stocks_df['Daily_Pct_Profit'],2)
    
    # Return the df
    return daily_stocks_df

In [8]:
def categorize_market_cap(market_cap_b):
    """
    Categorize companies based on market capitalization (in Billions).
    
    Categories:
    - Mega (>$200B)
    - Large ($10B-$200B)
    - Medium ($2B-$10B)
    - Small ($300M-$2B)
    - Micro ($50M-$300M)
    - Nano (<$50M)
    
    :param market_cap_b: Market capitalization in billions
    :return: Market cap category as a string
    """
    if pd.isna(market_cap_b) or market_cap_b <= 0:
        return "Unknown"

    if market_cap_b >= 200:
        return "Mega"
    elif 10 <= market_cap_b < 200:
        return "Large"
    elif 2 <= market_cap_b < 10:
        return "Medium"
    elif 0.3 <= market_cap_b < 2:
        return "Small"
    elif 0.05 <= market_cap_b < 0.3:
        return "Micro"
    else:
        return "Nano"

In [9]:
def create_stock_info_table(stock_tickers, delay=1.5, limit=None, existing_df=None, owned_tickers=None, refresh_only_owned=False):
    """
    Creates a stock information table with a time delay between requests to prevent Yahoo Finance's rate limit.
    Includes runtime progress updates.

    Notes:
        - Market_Cap value is in unit of Billions $

    Args:
        stock_tickers (list): A list of unique stock tickers to pull information for from Yahoo Finance.
        delay (float): Time delay (seconds) between requests. Default: 1.5
        limit (int): Limit number of tickers for testing or batching. Default: None
        existing_df (pd.DataFrame): Optional existing DataFrame with 'Stock' and 'Last_Updated' columns to prioritize updates.
        owned_tickers (list): Optional list of owned stock tickers to prioritize.
        refresh_only_owned (bool): If True, only refresh owned tickers regardless of existing_df. Default: False

    Returns:
        pd.DataFrame: Stock information with valuation, fundamentals, metadata, and ownership status.
    """
    if owned_tickers is None:
        owned_tickers = []

    # If refresh_only_owned is True, override stock_tickers with owned_tickers
    if refresh_only_owned:
        stock_tickers = [t for t in owned_tickers if t in stock_tickers]
    elif existing_df is not None and 'Stock' in existing_df.columns and 'Last_Updated' in existing_df.columns:
        existing_df = existing_df.copy()
        existing_df['Last_Updated'] = pd.to_datetime(existing_df['Last_Updated'], errors='coerce')
        cutoff = datetime.today() - timedelta(days=7)

        existing_df['is_owned'] = existing_df['Stock'].isin(owned_tickers)
        existing_df['needs_refresh'] = (
            (existing_df['Last_Updated'].isna()) |
            (existing_df['Last_Updated'] < cutoff)
        ) & existing_df['is_owned']

        existing_df = existing_df.sort_values(
            by=['needs_refresh', 'is_owned', 'Last_Updated', 'Market_Cap'],
            ascending=[False, False, True, False]
        )

        stock_tickers = [t for t in existing_df['Stock'].tolist() if t in stock_tickers]
    else:
        stock_tickers = sorted(stock_tickers, key=lambda t: (t not in owned_tickers,))

    if limit:
        stock_tickers = stock_tickers[:limit]

    stock_info_data = []
    total = len(stock_tickers)
    start_time = time.time()

    for i, ticker in enumerate(stock_tickers, start=1):
        try:
            stock = yf.Ticker(ticker)
            stock_info = stock.info

            company_name = stock_info.get("longName")
            officers = stock_info.get("companyOfficers", [])
            ceo_name = officers[0].get("name", "N/A") if officers and isinstance(officers[0], dict) else "N/A"
            country = stock_info.get("country", "N/A")
            state = stock_info.get("state", "N/A")
            city = stock_info.get("city", "N/A")

            market_cap = stock_info.get("marketCap")
            market_cap_b = market_cap / 1e9 if market_cap else np.nan

            enterprise_value_b = stock_info.get("enterpriseValue") / 1e9 if stock_info.get("enterpriseValue") else np.nan
            ebitda_b = stock_info.get("ebitda") / 1e9 if stock_info.get("ebitda") else np.nan
            revenue_b = stock_info.get("totalRevenue") / 1e9 if stock_info.get("totalRevenue") else np.nan

            profit_margins = round(stock_info.get("profitMargins", 0), 3)
            operating_margins = round(stock_info.get("operatingMargins", 0), 2)
            return_on_assets = round(stock_info.get("returnOnAssets", 0), 3)
            return_on_equity = round(stock_info.get("returnOnEquity", 0), 3)

            debt_to_equity = round(stock_info.get("debtToEquity", 0), 2)
            free_cashflow_b = stock_info.get("freeCashflow") / 1e9 if stock_info.get("freeCashflow") else np.nan
            avg_volume_m = stock_info.get("averageVolume") / 1e6 if stock_info.get("averageVolume") else np.nan
            shares_outstanding_b = stock_info.get("sharesOutstanding") / 1e9 if stock_info.get("sharesOutstanding") else np.nan
            
            short_interest = round(stock_info.get("shortPercentOfFloat", 0), 3)
            institutional_holdings = round(stock_info.get("heldPercentInstitutions", 0), 4)
            pe_ratio = round(stock_info.get("trailingPE", 0), 2)
            pb_ratio = round(stock_info.get("priceToBook", 0), 2)
            dividend_yield = round(stock_info.get("dividendYield", 0), 4)
            payout_ratio = round(stock_info.get("payoutRatio", 0), 4)
            ex_div_date = datetime.fromtimestamp(stock_info.get("exDividendDate")).strftime("%Y-%m-%d") if stock_info.get("exDividendDate") else np.nan
            beta = round(stock_info.get("beta", 0), 2)
            sector = stock_info.get("sector", "N/A")
            industry = stock_info.get("industry", "N/A")

            auditRisk = stock_info.get("auditRisk", "N/A")
            boardRisk = stock_info.get("boardRisk", "N/A")
            compensationRisk = stock_info.get("compensationRisk", "N/A")
            shareHolderRightsRisk = stock_info.get("shareHolderRightsRisk", "N/A")
            overallRisk = stock_info.get("overallRisk", "N/A")

            target_high = stock_info.get("targetHighPrice")
            target_low = stock_info.get("targetLowPrice")
            target_mean = stock_info.get("targetMeanPrice")
            target_median = stock_info.get("targetMedianPrice")
            recommendation_mean = stock_info.get("recommendationMean")
            recommendation_key = stock_info.get("recommendationKey", "N/A")
            num_analysts = stock_info.get("numberOfAnalystOpinions")
            description = stock_info.get("longBusinessSummary", "N/A")
            last_updated = pd.to_datetime(datetime.today().date())
            is_owned = ticker in owned_tickers

            stock_info_data.append([
                ticker, company_name, ceo_name, country, state, city, market_cap_b, enterprise_value_b, ebitda_b, revenue_b, 
                profit_margins, operating_margins, return_on_assets, return_on_equity, debt_to_equity, free_cashflow_b, avg_volume_m, 
                shares_outstanding_b, short_interest, institutional_holdings, pe_ratio, pb_ratio, dividend_yield, payout_ratio, 
                ex_div_date, beta, sector, industry, auditRisk, boardRisk, compensationRisk, shareHolderRightsRisk, overallRisk,
                target_high, target_low, target_mean, target_median, recommendation_mean, recommendation_key, num_analysts,
                description, last_updated, is_owned
            ])

        except Exception as e:
            print(f"Error fetching data for {ticker}: {e}")

        time.sleep(delay)

    stocks_info_df = pd.DataFrame(
        stock_info_data,
        columns=[
            "Stock", "Company", "CEO", "Country", "State", "City", "Market Cap (B)", "Enterprise Value (B)",
            "EBITDA (B)", "Revenue (B)", "Profit Margins", "Operating Margins", "Return On Assets",
            "Return On Equity", "Debt To Equity", "Free Cashflow (B)", "Avg Volume (M)", "Shares Outstanding (B)",
            "Short Interest", "Institutional Holdings", "PE Ratio", "PB Ratio", "Dividend Yield", "Payout Ratio",
            "Dividend Ex Date", "Beta", "Sector", "Industry", "Audit Risk", "Board Risk", "Compensation Risk", 
            "Shareholder Rights Risk", "Overall Risk", "Target High Price", "Target Low Price",
            "Target Mean Price", "Target Median Price", "Recommendation Mean", "Recommendation Key",
            "No Analysts", "Description", "Last Updated", "Is Owned"
        ]
    )

    stocks_info_df["Market_Cap_Category"] = stocks_info_df["Market Cap (B)"].apply(categorize_market_cap)

    stocks_info_df.loc[stocks_info_df["Stock"] == "O", "Sector"] = "Real Estate Investment Trusts"
    stocks_info_df.loc[stocks_info_df["Stock"] == "AFCG", "Sector"] = "Real Estate Investment Trusts"
    stocks_info_df.loc[stocks_info_df["Stock"] == "PLD", "Sector"] = "Real Estate Investment Trusts"
    stocks_info_df.loc[stocks_info_df["Stock"] == "V", "Sector"] = "Finance"

    stocks_info_df.sort_values(["Is Owned", "Last Updated", "Market Cap (B)"], ascending=[False, False, False], inplace=True)

    print("🎉 Stock info table creation complete.\n")
    return stocks_info_df

In [10]:
def fetch_nasdaq_tickers():
    """
    Fetches stock tickers from the NASDAQ index.

    Returns:
        nasdaq_tickers (set): A set of unique stock tickers.
    """
    # Connect to the FTP server
    ftp = ftplib.FTP('ftp.nasdaqtrader.com')
    ftp.login()  # Anonymous login

    # Navigate to the directory containing the file
    ftp.cwd('/SymbolDirectory')

    # Retrieve the file
    r = []
    ftp.retrlines('RETR nasdaqlisted.txt', r.append)
    ftp.quit()

    # Join the lines and create a StringIO object
    data = '\n'.join(r)
    data_io = io.StringIO(data)

    # Read the data into a pandas DataFrame
    df = pd.read_csv(data_io, sep='|')

    # Filter out entries where the symbol is not available
    df = df[df['Symbol'].notna()]

    # Extract the list of NASDAQ tickers
    nasdaq_tickers = df['Symbol'].tolist()

    return nasdaq_tickers

In [11]:
def fetch_sp500_tickers(yahoo_format: bool = True,
                        max_retries: int = 3,
                        timeout: int = 15) -> List[str]:
    """
    Fetch S&P 500 tickers with robust error handling and fallbacks.

    - Uses requests with a real User-Agent to avoid 403.
    - Wraps HTML in StringIO to satisfy pandas read_html deprecation.
    - Flattens MultiIndex columns safely before parsing.
    - Falls back to maintained CSV lists if Wikipedia is blocked/unavailable.

    Args:
        yahoo_format: Convert dots to dashes for Yahoo (e.g., BRK.B -> BRK-B).
        max_retries:  Retries per URL with exponential backoff.
        timeout:      HTTP timeout per request (seconds).

    Returns:
        List[str]: unique tickers.
    """
    wiki_urls = [
        "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
        "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies?useskin=vector",
        "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies&printable=yes",
    ]
    fallback_csv_urls = [
        "https://datahub.io/core/s-and-p-500-companies/r/constituents.csv",
        "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/master/data/constituents.csv",
    ]
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "en-US,en;q=0.9",
        "Cache-Control": "no-cache",
        "Pragma": "no-cache",
    }

    def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [
                " ".join([str(part) for part in tup if part is not None]).strip()
                for tup in df.columns.to_list()
            ]
        else:
            df.columns = [str(c).strip() for c in df.columns]
        return df

    def _clean(series: pd.Series) -> List[str]:
        s = series.astype(str).str.strip()
        s = s.str.replace(r"\s*\[.*?\]\s*", "", regex=True)
        s = s[s.str.len() > 0]
        if yahoo_format:
            s = s.str.replace(".", "-", regex=False)
        return sorted(s.unique().tolist())

    session = requests.Session()
    session.headers.update(headers)
    backoff = 1.5
    last_err: Optional[Exception] = None

    # --- Primary: Wikipedia ---
    for url in wiki_urls:
        for attempt in range(1, max_retries + 1):
            try:
                resp = session.get(url, timeout=timeout)
                resp.raise_for_status()
                tables = pd.read_html(io.StringIO(resp.text), flavor="bs4")
                for tbl in tables:
                    tbl = _flatten_columns(tbl)
                    cols_lower = [c.lower() for c in tbl.columns]
                    if "symbol" in cols_lower:
                        return _clean(tbl.iloc[:, cols_lower.index("symbol")])
                    if "ticker" in cols_lower:
                        return _clean(tbl.iloc[:, cols_lower.index("ticker")])
                last_err = RuntimeError(f"Parsed {url} but no Symbol/Ticker column found.")
                break
            except Exception as e:
                last_err = e
                if attempt < max_retries:
                    time.sleep(backoff ** attempt)
                else:
                    break

    # --- Fallback CSV ---
    for csv_url in fallback_csv_urls:
        try:
            df = pd.read_csv(csv_url, dtype=str)
            df = _flatten_columns(df)
            for col in ("Symbol", "Ticker", "symbol", "ticker"):
                if col in df.columns:
                    return _clean(df[col])
            return _clean(df.iloc[:, 0])
        except Exception as e:
            last_err = e
            continue

    raise RuntimeError(f"Failed to fetch S&P 500 tickers from all sources. Last error: {last_err}")

In [12]:
def rank_within_industry(df, column):
    """Ranks stocks within their industry for a given column (higher values are better)."""
    df[column + '_Rank'] = df.groupby('Industry')[column].rank(ascending=False, pct=True)
    return df

In [13]:
def calculate_buying_opportunity_scores(df, portfolio_cash=0):
    """
    Calculate buying opportunity scores and grades for each stock in the portfolio.
    
    This function evaluates each stock based on a weighted composite score built from:
    - Position relative to its 52-week high/low (lower is better)
    - Position size in the portfolio (smaller allocations score higher)
    - Sector and industry concentration to promote diversification
    - Risk factors (lower audit, board, compensation, and shareholder risks preferred)
    - Analyst target price discounts (greater discount to target = better opportunity)
    - Market sentiment score based on VIX and S&P 500 performance
    - Optional cash weighting (more available cash increases buying urgency)
    
    The output includes individual component scores, final composite score out of 100, and a qualitative grade
    ("strong_buy", "buy", "neutral", "sell", or "strong_sell").
    
    Args:
        df (pd.DataFrame): DataFrame containing stock portfolio and financial metrics.
        cash_position (float, optional): Cash available in the portfolio to weight buy urgency.
        vix (float, optional): VIX (fear index) value. Higher values increase buying opportunity.
        sp500_performance (float, optional): 30-day performance of the S&P 500 (used to calculate sentiment).
    
    Returns:
        pd.DataFrame: Original DataFrame with added columns for each score component, final score, and grade.
    """

    df = df.copy()

    # Fill key columns with safe defaults to prevent errors
    df["Price"] = pd.to_numeric(df["Price"], errors="coerce").fillna(0)
    df["52_Week_High"] = pd.to_numeric(df["52_Week_High"], errors="coerce").fillna(df["Price"])
    df["52_Week_Low"] = pd.to_numeric(df["52_Week_Low"], errors="coerce").fillna(df["Price"])
    df["Portfolio_Diversity"] = pd.to_numeric(df["Portfolio_Diversity"], errors="coerce").fillna(0)
    df["Target_Mean_Price"] = pd.to_numeric(df["Target Mean Price"], errors="coerce").fillna(df["Price"])

    # Risk score fields
    risk_cols = ["Audit Risk", "Board Risk", "Compensation Risk", "Shareholder Rights Risk", "Overall Risk"]
    for col in risk_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(3)  # Use 3 as a neutral/default risk level

    # Fetch market sentiment indicators
    try:
        vix = yf.Ticker("^VIX").history(period="1mo")["Close"].iloc[-1]
        sp500 = yf.Ticker("^GSPC").history(period="1mo")
        sp500_perf = (sp500["Close"].iloc[-1] - sp500["Close"].iloc[0]) / sp500["Close"].iloc[0]
    except Exception:
        vix = 20  # default VIX
        sp500_perf = 0  # neutral performance

    market_sentiment_score = (vix / 40) + (1 - sp500_perf)
    market_sentiment_score = np.clip(market_sentiment_score, 0, 1)

    # Normalize helper
    def normalize(series):
        min_val = series.min()
        max_val = series.max()
        return (series - min_val) / (max_val - min_val + 1e-6)

    # Compute component scores
    range_diff = (df["52_Week_High"] - df["52_Week_Low"]).replace(0, 1e-6)
    df["52wk_Score"] = 1 - (df["Price"] - df["52_Week_Low"]) / range_diff
    df["52wk_Score"] = df["52wk_Score"].clip(0, 1)

    df["Portfolio_Diversity_Score"] = 1 - normalize(df["Portfolio_Diversity"])

    target_discount = (df["Target_Mean_Price"] - df["Price"]) / df["Price"].replace(0, 1e-6)
    df["Target_Discount"] = target_discount.clip(0, 1)
    df["Target_Price_Score"] = normalize(df["Target_Discount"])

    df["Avg_Risk"] = df[risk_cols].mean(axis=1)
    df["Risk_Score"] = 1 - normalize(df["Avg_Risk"])

    # Add cash weighting bonus
    cash_weight_bonus = min(portfolio_cash / 10000, 1.0)

    # Weights
    weights = {
        "52wk_Score": 0.25,
        "Portfolio_Diversity_Score": 0.20,
        "Target_Price_Score": 0.20,
        "Risk_Score": 0.15,
        "Market_Sentiment_Score": 0.15,
        "Cash_Position_Score": 0.05
    }

    df["Buying_Opportunity_Score"] = (
        weights["52wk_Score"] * df["52wk_Score"] +
        weights["Portfolio_Diversity_Score"] * df["Portfolio_Diversity_Score"] +
        weights["Target_Price_Score"] * df["Target_Price_Score"] +
        weights["Risk_Score"] * df["Risk_Score"] +
        weights["Market_Sentiment_Score"] * market_sentiment_score +
        weights["Cash_Position_Score"] * cash_weight_bonus
    ) * 100

    def score_to_grade(score):
        if score >= 85: return "strong_buy"
        elif score >= 70: return "buy"
        elif score >= 50: return "neutral"
        elif score >= 30: return "sell"
        else: return "strong_sell"

    df["Buying_Grade"] = df["Buying_Opportunity_Score"].apply(score_to_grade)

    # Show key columns
    display_cols = [
        "Stock", "Price", "52_Week_High", "52_Week_Low", "Target_Mean_Price", "52wk_Score", "Portfolio_Diversity_Score", 
        "Target_Price_Score", "Risk_Score", "Buying_Opportunity_Score", "Buying_Grade"
    ]
    df = df[display_cols]

    # Round columns
    df["Risk_Score"] = round(df["Risk_Score"], 2)
    df["Target_Price_Score"] = round(df["Target_Price_Score"], 2)
    df["52wk_Score"] = round(df["52wk_Score"], 2)
    df["Buying_Opportunity_Score"] = round(df["Buying_Opportunity_Score"], 2)
    df["Portfolio_Diversity_Score"] = round(df["Portfolio_Diversity_Score"], 2)

    # Sort values by Buying_Opportunity_Score
    df = df.sort_values("Buying_Opportunity_Score", ascending=False)

    # Remove those without a price
    df = df[df['Price'] > 0]

    return df

## Build Stock Tables

### Stock Summary DF 

In [14]:
# Create Stock Summary df
stock_summary_df = build_summary_dataframe(stock_dictionary)

Error fetching data for WMT: Selling more shares than owned for WMT on 2022-10-13 00:00:00


$ROVR: possibly delisted; no timezone found


In [15]:
# Print out a sample
display(stock_summary_df.head(10))
# Save the df to the data folder
stock_summary_df.to_csv('data/stock_summary.csv')

,Stock,Company,Price,Quantity,Avg_Cost,Market_Value,Percent_Change,Equity_Change,52_Week_High,52_Week_Low,Portfolio_Diversity,Direction,Last_Updated
54,BE,Bloom Energy Corporation,105.0000,100.0000,13.1600,10500.0000,299.5400,9184.2200,147.8600,15.1500,10.6700,Up,2025-12-03
7,NVDA,NVIDIA Corporation,181.4600,45.0000,14.5700,8165.7000,25.0200,7510.2600,212.1900,86.6200,8.3000,Up,2025-12-03
78,ASML,ASML Holding N.V.,1108.7800,5.0000,727.1400,5543.9000,54.0100,1908.2100,1113.2100,578.5100,5.6300,Up,2025-12-03
31,AMD,"Advanced Micro Devices, Inc.",215.2400,20.0000,113.2400,4304.8000,49.4800,2039.9100,267.0800,76.4800,4.3700,Up,2025-12-03
61,MELI,"MercadoLibre, Inc.",2115.9100,2.0000,1528.4800,4231.8200,6.4600,1174.8600,2645.2200,1646.0000,4.3000,Up,2025-12-03
75,BEPC,Brookfield Renewable Corporation,40.4100,100.0000,27.9300,4041.0000,30.7800,1248.0800,45.1000,23.7300,4.1100,Up,2025-12-03
50,SNOW,Snowflake Inc.,259.6800,13.0000,156.2200,3375.8400,39.2300,1345.0100,280.6700,120.1000,3.4300,Up,2025-12-03
68,GOOG,Alphabet Inc.,316.0200,10.0000,137.9700,3160.2000,79.4700,1780.4600,328.6700,142.6600,3.2100,Up,2025-12-03
15,UBER,"Uber Technologies, Inc.",87.5700,36.0000,49.3100,3152.5200,21.3600,1377.4500,101.9900,59.3300,3.2000,Up,2025-12-03
14,FSLR,"First Solar, Inc.",262.5600,11.0000,84.0600,2888.1600,30.2600,1963.5500,281.5500,116.5600,2.9300,Up,2025-12-03


### Daily Stocks DF 

In [16]:
# Create Daily Stocks df
daily_stocks_df = create_daily_stock_table(stock_dictionary)

$ROVR: possibly delisted; no timezone found


Error fetching data for ROVR: Can only use .dt accessor with datetimelike values


In [17]:
# Print out a sample
display(daily_stocks_df.tail(10))
# Save the df to the data folder
daily_stocks_df.to_csv('data/daily_stocks.csv')

,Date,Close,Stock,Shares_Held,Avg_Cost,Equity,Market_Value,Total_Profit,Daily_Profit,Daily_Pct_Profit
200906,2025-11-18,54.0400,VWO,75.0000,44.5400,3340.6700,4053.0000,712.3300,-13.5000,-0.3300
200907,2025-11-19,53.9200,VWO,75.0000,44.5400,3340.6700,4044.0000,703.3300,-9.0000,-0.2200
200908,2025-11-20,53.2800,VWO,75.0000,44.5400,3340.6700,3996.0000,655.3300,-48.0000,-1.1900
200909,2025-11-21,53.0700,VWO,75.0000,44.5400,3340.6700,3980.2500,639.5800,-15.7500,-0.3900
200910,2025-11-24,53.6100,VWO,75.0000,44.5400,3340.6700,4020.7500,680.0800,40.5000,1.0200
200911,2025-11-25,53.8500,VWO,75.0000,44.5400,3340.6700,4038.7500,698.0800,18.0000,0.4500
200912,2025-11-26,54.1100,VWO,75.0000,44.5400,3340.6700,4058.2500,717.5800,19.5000,0.4800
200913,2025-11-28,54.3000,VWO,75.0000,44.5400,3340.6700,4072.5000,731.8300,14.2500,0.3500
200914,2025-12-01,54.2500,VWO,75.0000,44.5400,3340.6700,4068.7500,728.0800,-3.7500,-0.0900
200915,2025-12-02,54.1300,VWO,75.0000,44.5400,3340.6700,4059.7500,719.0800,-9.0000,-0.2200


### Stock Info DF

In [18]:
# Fetch the list of Nasdaq tickers
nasdaq_tickers = fetch_nasdaq_tickers()

# Fetch the list of SP500 tickers
sp500_tickers = fetch_sp500_tickers()

# Combine all tickers into a set to remove duplicates
unique_stock_tickers = list(set(sp500_tickers + nasdaq_tickers))

# Get a list of my owned stock tickers
my_tickers = stock_summary['Stock'].tolist()

In [19]:
nasdaq_tickers = set(fetch_nasdaq_tickers())  # Convert list to set
sp500_tickers = set(fetch_sp500_tickers())  # Convert list to set

# Find overlap
sp500_in_nasdaq = sp500_tickers.intersection(nasdaq_tickers)
sp500_not_in_nasdaq = sp500_tickers - nasdaq_tickers

# Print results
print(f"Total S&P 500 Tickers: {len(sp500_tickers)}")
print(f"Total Nasdaq Tickers: {len(nasdaq_tickers)}")
print(f"Number of S&P 500 companies also in Nasdaq: {len(sp500_in_nasdaq)}")
print(f"Number of S&P 500 companies NOT in Nasdaq: {len(sp500_not_in_nasdaq)}")

Total S&P 500 Tickers: 503
Total Nasdaq Tickers: 5220
Number of S&P 500 companies also in Nasdaq: 161
Number of S&P 500 companies NOT in Nasdaq: 342


In [20]:
# Create Stock Info df
stock_info_df = create_stock_info_table(unique_stock_tickers, limit=100, owned_tickers=my_tickers, refresh_only_owned=True)
# Print out a sample
display(stock_info_df.head(20))

🎉 Stock info table creation complete.



,Stock,Company,CEO,Country,State,City,Market Cap (B),Enterprise Value (B),EBITDA (B),Revenue (B),Profit Margins,Operating Margins,Return On Assets,Return On Equity,Debt To Equity,Free Cashflow (B),Avg Volume (M),Shares Outstanding (B),Short Interest,Institutional Holdings,PE Ratio,PB Ratio,Dividend Yield,Payout Ratio,Dividend Ex Date,Beta,Sector,Industry,Audit Risk,Board Risk,Compensation Risk,Shareholder Rights Risk,Overall Risk,Target High Price,Target Low Price,Target Mean Price,Target Median Price,Recommendation Mean,Recommendation Key,No Analysts,Description,Last Updated,Is Owned,Market_Cap_Category
0,NVDA,NVIDIA Corporation,Mr. Jen-Hsun Huang,United States,CA,Santa Clara,4425.3661,4359.6920,112.6960,187.1420,0.5300,0.6300,0.5350,1.0740,9.1000,53.2829,191.3703,24.3470,0.0100,0.6932,44.8000,37.0900,0.0200,0.0099,2025-12-03,2.2700,Technology,Semiconductors,5,10,4,8,8,352.0000,140.0000,250.6614,250.0000,1.3281,strong_buy,56,"NVIDIA Corporation, a computing infrastructure...",2025-12-03,True,Mega
26,AAPL,Apple Inc.,Mr. Timothy D. Cook,United States,CA,Cupertino,4247.1711,4286.5244,144.7480,416.1610,0.2690,0.3200,0.2300,1.7140,152.4100,78.8623,51.2692,14.7764,0.0080,0.6436,38.3100,57.3400,0.3600,0.1367,2025-11-09,1.1100,Technology,Consumer Electronics,7,1,3,1,1,345.0000,215.0000,281.9919,280.0000,2.0000,buy,41,"Apple Inc. designs, manufactures, and markets ...",2025-12-03,True,Mega
7,GOOG,Alphabet Inc.,Mr. Sundar Pichai,United States,CA,Mountain View,3814.9407,3759.1125,145.1740,385.4760,0.3220,0.3100,0.1630,0.3540,11.4200,47.9978,23.6953,5.4070,0.0000,0.6076,31.1700,9.8700,0.2700,0.0809,2025-12-07,1.0800,Communication Services,Internet Content & Information,N/A,N/A,N/A,N/A,N/A,380.0000,185.0000,314.8059,330.0000,1.4769,strong_buy,17,Alphabet Inc. offers various products and plat...,2025-12-03,True,Mega
10,MSFT,Microsoft Corporation,Mr. Satya Nadella,United States,WA,Redmond,3642.2515,3660.2281,166.4370,293.8120,0.3570,0.4900,0.1470,0.3220,33.1500,53.3274,21.9205,7.4324,0.0060,0.7568,34.8500,10.0300,0.7400,0.2361,2025-11-19,1.0600,Technology,Software - Infrastructure,9,8,5,2,5,730.0000,483.0000,625.4096,634.1500,1.2500,strong_buy,52,Microsoft Corporation develops and supports so...,2025-12-03,True,Mega
18,V,Visa Inc.,Mr. Ryan M. McInerney,United States,CA,San Francisco,639.7732,638.8141,28.0280,40.0000,0.5010,0.6600,0.1730,0.5210,68.8100,20.0729,6.1392,1.6876,0.0130,0.9064,32.3500,17.0100,0.8100,0.2314,2025-11-11,0.8200,Finance,Credit Services,10,1,1,3,2,450.0000,305.0000,394.4313,405.0000,1.6000,buy,37,Visa Inc. operates as a payment technology com...,2025-12-03,True,Mega
27,JNJ,Johnson & Johnson,Mr. Joaquin Duato,United States,NJ,New Brunswick,494.9174,522.1504,31.9220,92.1490,0.2730,0.3000,0.0830,0.3360,57.7700,12.0090,8.7433,2.4093,0.0080,0.7504,19.8500,6.2300,2.5300,0.4908,2025-11-24,0.3600,Healthcare,Drug Manufacturers - General,6,8,3,3,4,230.0000,155.0000,202.5433,205.5000,2.1600,buy,24,"Johnson & Johnson, together with its subsidiar...",2025-12-03,True,Mega
1,ASML,ASML Holding N.V.,Mr. Christophe D. Fouquet,Netherlands,N/A,Veldhoven,430.3704,427.0070,12.1579,32.2120,0.2940,0.3300,0.1620,0.5390,14.2400,9.3203,1.5801,0.3881,0.0040,0.1857,39.3700,22.6100,0.6600,0.2672,2025-10-28,1.3500,Technology,Semiconductor Equipment & Materials,N/A,N/A,N/A,N/A,N/A,1338.9226,755.2520,1063.1434,986.8627,1.7368,buy,13,ASML Holding N.V. provides lithography solutio...,2025-12-03,True,Mega
8,COST,Costco Wholesale Corporation,Mr. Ron M. Vachris,United States,WA,Issaquah,408.8992,403.2755,12.8090,275.2350,0.0290,0.0400,0.0880,0.3070,34.0700,5.8991,2.4365,0.4432,0.0160,0.7209,50.7700,14.0100,0.5600,0.2702,2025-10-30,0.9800,Consumer Defensive,Discount Stores,6,3,5,1,1,1218.0000,640.0000,1059.8334,1065.0000,1.9722,buy,30,"Costco Wholesale Corporation, together with it...",2025-12-03,True,Mega
20,ABBV,AbbVie Inc.,Mr. Robert A. Michael CPA,United States,IL,North Chicago,396.5481,459.7691,29.5190,59.6440,0.0400,0.3500,0.0960,1.3800,0.0000,20.7990,

In [21]:
# Save the df to the data folder
stock_info_df.to_csv('data/stock_info.csv')

### Create Final Tables by merging and cleaning dfs 

In [22]:
## Create a lookup table for Stock Tickers to Company Names
stock_names = stock_summary_df[['Stock', 'Company']]

In [23]:
# Drop overlapping columns from stock_summary_df before merge
columns_to_drop = [col for col in stock_summary_df.columns if col in stock_info_df.columns and col != "Stock"]
stock_summary_df_clean = stock_summary_df.drop(columns=columns_to_drop)
## Merge stocks and stocks_info
stocks_complete = pd.merge(stock_summary_df_clean, stock_info_df, how = 'left', on = 'Stock')
stocks_complete.head()

,Stock,Price,Quantity,Avg_Cost,Market_Value,Percent_Change,Equity_Change,52_Week_High,52_Week_Low,Portfolio_Diversity,Direction,Last_Updated,Company,CEO,Country,State,City,Market Cap (B),Enterprise Value (B),EBITDA (B),Revenue (B),Profit Margins,Operating Margins,Return On Assets,Return On Equity,Debt To Equity,Free Cashflow (B),Avg Volume (M),Shares Outstanding (B),Short Interest,Institutional Holdings,PE Ratio,PB Ratio,Dividend Yield,Payout Ratio,Dividend Ex Date,Beta,Sector,Industry,Audit Risk,Board Risk,Compensation Risk,Shareholder Rights Risk,Overall Risk,Target High Price,Target Low Price,Target Mean Price,Target Median Price,Recommendation Mean,Recommendation Key,No Analysts,Description,Last Updated,Is Owned,Market_Cap_Category
0,BE,105.0000,100.0000,13.1600,10500.0000,299.5400,9184.2200,147.8600,15.1500,10.6700,Up,2025-12-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
1,NVDA,181.4600,45.0000,14.5700,8165.7000,25.0200,7510.2600,212.1900,86.6200,8.3000,Up,2025-12-03,NVIDIA Corporation,Mr. Jen-Hsun Huang,United States,CA,Santa Clara,4425.3661,4359.6920,112.6960,187.1420,0.5300,0.6300,0.5350,1.0740,9.1000,53.2829,191.3703,24.3470,0.0100,0.6932,44.8000,37.0900,0.0200,0.0099,2025-12-03,2.2700,Technology,Semiconductors,5,10,4,8,8,352.0000,140.0000,250.6614,250.0000,1.3281,strong_buy,56.0000,"NVIDIA Corporation, a computing infrastructure...",2025-12-03,True,Mega
2,ASML,1108.7800,5.0000,727.1400,5543.9000,54.0100,1908.2100,1113.2100,578.5100,5.6300,Up,2025-12-03,ASML Holding N.V.,Mr. Christophe D. Fouquet,Netherlands,N/A,Veldhoven,430.3704,427.0070,12.1579,32.2120,0.2940,0.3300,0.1620,0.5390,14.2400,9.3203,1.5801,0.3881,0.0040,0.1857,39.3700,22.6100,0.6600,0.2672,2025-10-28,1.3500,Technology,Semiconductor Equipment & Materials,N/A,N/A,N/A,N/A,N/A,1338.9226,755.2520,1063.1434,986.8627,1.7368,buy,13.0000,ASML Holding N.V. provides lithography solutio...,2025-12-03,True,Mega
3,AMD,215.2400,20.0000,113.2400,4304.8000,49.4800,2039.9100,267.0800,76.4800,4.3700,Up,2025-12-03,"Advanced Micro Devices, Inc.",Dr. Lisa T. Su Ph.D.,United States,CA,Santa Clara,350.4197,347.0467,6.0540,32.0270,0.1030,0.1400,0.0260,0.0530,6.3700,3.2455,59.1954,1.6280,0.0250,0.6969,112.1000,5.7600,0.0000,0.0000,1995-04-26,1.9100,Technology,Semiconductors,7,4,9,3,6,380.0000,178.0000,283.5651,290.0000,1.5490,buy,43.0000,"Advanced Micro Devices, Inc. operates as a sem...",2025-12-03,True,Mega
4,MELI,2115.9100,2.0000,1528.4800,4231.8200,6.4600,1174.8600,2645.2200,1646.0000,4.3000,Up,2025-12-03,"MercadoLibre, Inc.",Mr. Marcos Eduardo Galperín,Uruguay,N/A,Montevideo,107.2707,113.0817,3.8640,26.1930,0.0790,0.1000,0.0660,0.4060,159.3000,-4.0662,0.5246,0.0507,0.0160,0.8448,51.5400,17.1700,0.0000,0.0000,2017-12-27,1.4200,Consumer Cyclical,Internet Retail,7,9,5,7,8,3500.0000,2190.0000,2847.3462,2842.5000,1.4231,strong_buy,26.0000,"MercadoLibre, Inc. operates online commerce pl...",2025-12-03,True,Large


In [24]:
stocks_complete.columns

Index(['Stock', 'Price', 'Quantity', 'Avg_Cost', 'Market_Value', 'Percent_Change', 'Equity_Change', '52_Week_High', '52_Week_Low', 'Portfolio_Diversity', 'Direction', 'Last_Updated', 'Company', 'CEO', 'Country', 'State', 'City', 'Market Cap (B)', 'Enterprise Value (B)', 'EBITDA (B)', 'Revenue (B)', 'Profit Margins', 'Operating Margins', 'Return On Assets', 'Return On Equity', 'Debt To Equity', 'Free Cashflow (B)', 'Avg Volume (M)', 'Shares Outstanding (B)', 'Short Interest', 'Institutional Holdings', 'PE Ratio', 'PB Ratio', 'Dividend Yield', 'Payout Ratio', 'Dividend Ex Date', 'Beta', 'Sector', 'Industry', 'Audit Risk', 'Board Risk', 'Compensation Risk', 'Shareholder Rights Risk', 'Overall Risk', 'Target High Price', 'Target Low Price', 'Target Mean Price', 'Target Median Price', 'Recommendation Mean', 'Recommendation Key', 'No Analysts', 'Description', 'Last Updated', 'Is Owned', 'Market_Cap_Category'], dtype='object')

In [25]:
## Merge daily_stocks and stock_names
daily_stocks_complete = pd.merge(daily_stocks_df, stock_names)
# Create datetime column
daily_stocks_complete['Datetime'] = pd.to_datetime(daily_stocks_complete['Date'])

In [26]:
## Create a dataframe with daily total equity
daily_equity = daily_stocks_complete.groupby(by=["Date"])[["Market_Value", "Equity", "Total_Profit"]].sum()
daily_equity = daily_equity.reset_index()
# Set Date column as datetime
daily_equity['Date'] = pd.to_datetime(daily_equity['Date'])
# Create Daily_Profit column
daily_equity['Daily_Profit'] = daily_equity['Total_Profit'].diff()
# Format Daily_Profit columns
daily_equity['Daily_Profit'] = round(daily_equity['Daily_Profit'], 2)
# Remove unnamed columns
daily_equity = daily_equity.loc[:, ~daily_equity.columns.str.contains('^Unnamed')]

In [27]:
## Create Daily Gainers / Losers dataframes
most_recent_date = daily_stocks_complete['Datetime'].max()
todays_stocks = daily_stocks_complete[(daily_stocks_complete['Datetime'] == most_recent_date) &
                                      (daily_stocks_complete['Shares_Held'] != 0) &
                                      (daily_stocks_complete['Company'] != "0")].copy()
daily_gainers = todays_stocks[['Company', 'Daily_Profit', 'Daily_Pct_Profit']].reset_index(drop=True).sort_values("Daily_Profit", axis = 0, ascending = False).head(5)
daily_losers = todays_stocks[['Company', 'Daily_Profit', 'Daily_Pct_Profit']].reset_index(drop=True).sort_values("Daily_Profit", axis = 0, ascending = True).head(5)
# Remove any negatives from gainers and positives from losers
daily_gainers = daily_gainers[daily_gainers['Daily_Profit'] > 0]
daily_losers = daily_losers[daily_losers['Daily_Profit'] < 0]
# Format dataframe values
daily_gainers['Daily_Profit'] = daily_gainers['Daily_Profit'].apply(lambda x: "${:,.2f}".format(x))
daily_gainers['Daily_Pct_Profit'] = daily_gainers['Daily_Pct_Profit'].apply(lambda x: "{:.2f}%".format(x))
daily_losers['Daily_Profit'] = daily_losers['Daily_Profit'].apply(lambda x: "${:,.2f}".format(x))
daily_losers['Daily_Pct_Profit'] = daily_losers['Daily_Pct_Profit'].apply(lambda x: "{:.2f}%".format(x))

## Buying Opportunities

I also want to have a smarter way to re-buy into stocks that I currently own. To do this, I will create a "buy score", which will consist of the following 4 criteria:

- % of total portfolio (Value based on ranked position)
- stock's industry representation (Value based on ranked position)
- 52-week high low value (the lower in the 52 week high low, the higher the score)
- Yahoo Finance price target and buy recommendation

Each category will get a value assigned between 0-25 according to its position in the range of value. This will be done by comparing the value to the maximum and minumum for the category to compute its relative position. The resulting value will then get scaled to the 0-25 scale and the scores from each category will then be summed to create a final "buy score". By comparing the buy scores across all stocks, we can see which are the best opportunities to deploy our available money.

In [28]:
# Calculate how much a stock is trading below its target price
stocks_complete["Target_Discount"] = (
    (stocks_complete["Target Mean Price"] - stocks_complete["Price"]) / stocks_complete["Target Mean Price"]
)
 # Ensure it's between 0-100%
stocks_complete["Target_Discount"] = stocks_complete["Target_Discount"].clip(0, 1)

In [29]:
# Fetch VIX (Fear Index)
vix = yf.Ticker("^VIX").history(period="1mo")["Close"].iloc[-1]

# Fetch S&P 500 performance (30 days)
sp500 = yf.Ticker("^GSPC").history(period="1mo")
sp500_performance = (sp500["Close"].iloc[-1] - sp500["Close"].iloc[0]) / sp500["Close"].iloc[0]

# Assign Market Sentiment Score (Inverse relationship)
market_sentiment_score = (vix / 40) + (1 - sp500_performance)  # Higher VIX and lower SP500 → higher score
# market_sentiment_score = np.clip(market_sentiment_score, 0, 1)

# Add this market sentiment score as a new columns
stocks_complete["Market_Sentiment_Score"] = market_sentiment_score

In [30]:
stocks_complete.head()

,Stock,Price,Quantity,Avg_Cost,Market_Value,Percent_Change,Equity_Change,52_Week_High,52_Week_Low,Portfolio_Diversity,Direction,Last_Updated,Company,CEO,Country,State,City,Market Cap (B),Enterprise Value (B),EBITDA (B),Revenue (B),Profit Margins,Operating Margins,Return On Assets,Return On Equity,Debt To Equity,Free Cashflow (B),Avg Volume (M),Shares Outstanding (B),Short Interest,Institutional Holdings,PE Ratio,PB Ratio,Dividend Yield,Payout Ratio,Dividend Ex Date,Beta,Sector,Industry,Audit Risk,Board Risk,Compensation Risk,Shareholder Rights Risk,Overall Risk,Target High Price,Target Low Price,Target Mean Price,Target Median Price,Recommendation Mean,Recommendation Key,No Analysts,Description,Last Updated,Is Owned,Market_Cap_Category,Target_Discount,Market_Sentiment_Score
0,BE,105.0000,100.0000,13.1600,10500.0000,299.5400,9184.2200,147.8600,15.1500,10.6700,Up,2025-12-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,1.4138
1,NVDA,181.4600,45.0000,14.5700,8165.7000,25.0200,7510.2600,212.1900,86.6200,8.3000,Up,2025-12-03,NVIDIA Corporation,Mr. Jen-Hsun Huang,United States,CA,Santa Clara,4425.3661,4359.6920,112.6960,187.1420,0.5300,0.6300,0.5350,1.0740,9.1000,53.2829,191.3703,24.3470,0.0100,0.6932,44.8000,37.0900,0.0200,0.0099,2025-12-03,2.2700,Technology,Semiconductors,5,10,4,8,8,352.0000,140.0000,250.6614,250.0000,1.3281,strong_buy,56.0000,"NVIDIA Corporation, a computing infrastructure...",2025-12-03,True,Mega,0.2761,1.4138
2,ASML,1108.7800,5.0000,727.1400,5543.9000,54.0100,1908.2100,1113.2100,578.5100,5.6300,Up,2025-12-03,ASML Holding N.V.,Mr. Christophe D. Fouquet,Netherlands,N/A,Veldhoven,430.3704,427.0070,12.1579,32.2120,0.2940,0.3300,0.1620,0.5390,14.2400,9.3203,1.5801,0.3881,0.0040,0.1857,39.3700,22.6100,0.6600,0.2672,2025-10-28,1.3500,Technology,Semiconductor Equipment & Materials,N/A,N/A,N/A,N/A,N/A,1338.9226,755.2520,1063.1434,986.8627,1.7368,buy,13.0000,ASML Holding N.V. provides lithography solutio...,2025-12-03,True,Mega,0.0000,1.4138
3,AMD,215.2400,20.0000,113.2400,4304.8000,49.4800,2039.9100,267.0800,76.4800,4.3700,Up,2025-12-03,"Advanced Micro Devices, Inc.",Dr. Lisa T. Su Ph.D.,United States,CA,Santa Clara,350.4197,347.0467,6.0540,32.0270,0.1030,0.1400,0.0260,0.0530,6.3700,3.2455,59.1954,1.6280,0.0250,0.6969,112.1000,5.7600,0.0000,0.0000,1995-04-26,1.9100,Technology,Semiconductors,7,4,9,3,6,380.0000,178.0000,283.5651,290.0000,1.5490,buy,43.0000,"Advanced Micro Devices, Inc. operates as a sem...",2025-12-03,True,Mega,0.2410,1.4138
4,MELI,2115.9100,2.0000,1528.4800,4231.8200,6.4600,1174.8600,2645.2200,1646.0000,4.3000,Up,2025-12-03,"MercadoLibre, Inc.",Mr. Marcos Eduardo Galperín,Uruguay,N/A,Montevideo,107.2707,113.0817,3.8640,26.1930,0.0790,0.1000,0.0660,0.4060,159.3000,-4.0662,0.5246,0.0507,0.0160,0.8448,51.5400,17.1700,0.0000,0.0000,2017-12-27,1.4200,Consumer Cyclical,Internet Retail,7,9,5,7,8,3500.0000,2190.0000,2847.3462,2842.5000,1.4231,strong_buy,26.0000,"MercadoLibre, Inc. operates online commerce pl...",2025-12-03,True,Large,0.2569,1.4138


In [31]:
stocks_complete.columns

Index(['Stock', 'Price', 'Quantity', 'Avg_Cost', 'Market_Value', 'Percent_Change', 'Equity_Change', '52_Week_High', '52_Week_Low', 'Portfolio_Diversity', 'Direction', 'Last_Updated', 'Company', 'CEO', 'Country', 'State', 'City', 'Market Cap (B)', 'Enterprise Value (B)', 'EBITDA (B)', 'Revenue (B)', 'Profit Margins', 'Operating Margins', 'Return On Assets', 'Return On Equity', 'Debt To Equity', 'Free Cashflow (B)', 'Avg Volume (M)', 'Shares Outstanding (B)', 'Short Interest', 'Institutional Holdings', 'PE Ratio', 'PB Ratio', 'Dividend Yield', 'Payout Ratio', 'Dividend Ex Date', 'Beta', 'Sector', 'Industry', 'Audit Risk', 'Board Risk', 'Compensation Risk', 'Shareholder Rights Risk', 'Overall Risk', 'Target High Price', 'Target Low Price', 'Target Mean Price', 'Target Median Price', 'Recommendation Mean', 'Recommendation Key', 'No Analysts', 'Description', 'Last Updated', 'Is Owned', 'Market_Cap_Category', 'Target_Discount', 'Market_Sentiment_Score'], dtype='object')

In [32]:
portfolio_cash = 12275

buying_opportunities = calculate_buying_opportunity_scores(stocks_complete, portfolio_cash)
buying_opportunities

,Stock,Price,52_Week_High,52_Week_Low,Target_Mean_Price,52wk_Score,Portfolio_Diversity_Score,Target_Price_Score,Risk_Score,Buying_Opportunity_Score,Buying_Grade
43,AFCG,2.8800,9.7100,2.5200,7.3333,0.9500,0.9800,1.0000,0.2000,86.2600,strong_buy
47,OTLY,11.9300,18.8400,6.0000,20.1667,0.5400,0.9900,0.6900,0.8300,79.5900,buy
40,VICI,28.4500,34.0300,27.9800,35.9091,0.9200,0.9500,0.2600,0.7600,78.5600,buy
24,DUOL,182.6100,544.9300,166.2700,270.1137,0.9600,0.8600,0.4800,0.2900,75.1200,buy
25,TREX,34.8500,80.7400,29.7700,34.8500,0.9000,0.8800,0.0000,0.8300,72.4900,buy
15,TTD,39.9500,141.5300,38.2200,62.3333,0.9800,0.7800,0.5600,0.0000,71.3800,buy
10,MRNA,24.0600,48.9200,22.2800,35.7778,0.9300,0.7300,0.4900,0.1700,70.1400,buy
36,WDAY,213.0600,294.0000,205.3300,275.6366,0.9100,0.9400,0.2900,0.1500,69.6700,neutral
21,CMG,34.1400,66.7400,29.7500,43.1818,0.8800,0.8100,0.2600,0.4100,69.6500,neutral
18,VZ,40.6100,47.3600,37.5900,47.5250,0.6900,0.7900,0.1700,0.8500,69.2300,neutral
